# Overflow prediction: saturation/context vs. linear probes

This notebook builds two **ROC-AUC comparison tables** (TriviaQA, SQuAD, HotpotQA) for overflow prediction:

- **Table 1 — Main results:** Pre-compression (context only), Pre-inference (embedding, saturation), Post-inference (attention, embedding, saturation).
- **Table 2 — Internal ablation:** Pre/Post-projection, Middle layer, Last layer (embedding, saturation, attention).

**Setup:** Set `BASE_PATH` and `FEATURES_PATH_TEMPLATE` in the config cell, then run all. Tables are produced by `generate_tables()` and shown in the notebook (best = **bold**, second-best = *underline*). Metric: ROC-AUC, 5-fold CV.

In [1]:
# Config: set paths before running (use {} for dataset: trivia, squad, hotpot)
BASE_PATH = "/app/xlong/scripts/data_preprocessing/runs/squad"                    # root dir with mistral_trivia, mistral_squad, mistral_hotpot
FEATURES_PATH_TEMPLATE = f"{BASE_PATH}/features_context_metrics.jsonl"  # or features_interpretation.jsonl

In [5]:
import json
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate


def interprete_features(filepath, name):
    df_features = load_dataset(filepath.format(name))
    plot_interpretation(df_features)


def load_dataset(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))

    df_features = pd.json_normalize(rows, sep=".")
    print(len(df_features))
    return df_features

In [6]:
# ============================================================================
# TABLES: MAIN RESULTS + INTERNAL ABLATION (displayed in notebook)
# ============================================================================

import os
from pathlib import Path
import numpy as np

DATA_COLS = ['TriviaQA', 'SQuAD', 'HotpotQA']


def load_probing_results(dataset_name, result_type):
    """
    Load probing results from JSON files
    
    Args:
        dataset_name: 'trivia', 'hotpot', or 'squad'
        result_type: 'xrag_no_query', 'setting1', or 'setting2'
    
    Returns:
        dict: Parsed JSON results or None if file not found
    """
    base_path = os.path.join(BASE_PATH, "results")

    file_mapping = {
        'xrag_no_query': 'probing_results_no_query_combined.json',
        'setting1': 'probing_results_setting1_probing.json',
        'setting2': 'probing_results_setting2_probing.json'
    }
    
    filepath = os.path.join(base_path, file_mapping[result_type])
    
    if not os.path.exists(filepath):
        return None
    
    with open(filepath, 'r') as f:
        return json.load(f)

def style_auc_table(df, df_means, data_cols=DATA_COLS):
    """Apply bold (best) and underline (second) per dataset column."""
    style_df = pd.DataFrame('', index=df.index, columns=df.columns)
    for col in data_cols:
        if col not in df_means.columns:
            continue
        vals = df_means[col].dropna()
        if len(vals) == 0:
            continue
        max_val = vals.max()
        second_val = vals.nlargest(2).iloc[1] if len(vals) >= 2 else None
        for i in df.index:
            v = df_means.loc[i, col]
            if pd.isna(v):
                continue
            if v == max_val:
                style_df.loc[i, col] = 'font-weight: bold'
            elif second_val is not None and v == second_val:
                style_df.loc[i, col] = 'text-decoration: underline'
    return style_df

def generate_tables():
    """
    Build data for two ROC-AUC comparison tables.
    1. Main Results: primary pipeline stages (Pre-compression, Pre-inference, Post-inference).
    2. Internal Ablation: internal LLM components (Pre-projection, Post-projection, Middle layer, Last layer).
    Returns (table1_df, table1_means), (table2_df, table2_means) for display and styling.
    """
    
    #datasets = ['trivia', 'squad', 'hotpot']
    datasets = ['squad']
    data_cols = DATA_COLS
    
    # Load all data
    all_data = {}
    
    for dataset in datasets:
        all_data[dataset] = {
            'probing_no_query': load_probing_results(dataset, 'xrag_no_query'),
            'probing_setting1': load_probing_results(dataset, 'setting1'),
            'probing_setting2': load_probing_results(dataset, 'setting2'),
            'features': None
        }
        
        # Load feature data
        filepath = FEATURES_PATH_TEMPLATE.format(dataset)
        df_features = load_dataset(filepath)
        print(df_features.columns)
        # Compute saturation features
        saturation_types = ['spec_entropy', 'excess_kurtosis', 'hoyer']
        
        context_features = ['context_perplexity', 'context_length_chars', 'context_gzip_bits_per_char', 'context_length_tokens']
        preproj_saturation_features = [col for col in df_features.columns.values if any([st in col for st in saturation_types]) and ('preproj_metrics' in col) and (col[-2:] != '_n')]
        postproj_saturation_features = [col for col in df_features.columns.values if any([st in col for st in saturation_types]) and ('postproj_metrics' in col) and (col[-2:] != '_n')]
        prellm_saturation_features = [col for col in df_features.columns.values if any([st in col for st in saturation_types]) and ('postproj_metrics' in col or 'preproj_metrics' in col) and (col[-2:] != '_n')]
        mid_saturation_xrag_only = [col for col in df_features.columns.values if any([st in col for st in saturation_types]) and ('mid_metrics' in col) and ('nonxrag' not in col) and (col[-2:] != '_n')]
        mid_saturation_with_nonxrag = [col for col in df_features.columns.values if any([st in col for st in saturation_types]) and ('nonxrag' in col) and ('mid_group_metrics' in col) and (col[-2:] != '_n') and ('_first' not in col) and ('_last' not in col)] + mid_saturation_xrag_only
        mid_attn_features = [col for col in df_features.columns.values if ('attn_mid' in col)]
        last_saturation_xrag_only = [col for col in df_features.columns.values if any([st in col for st in saturation_types]) and ('last_metrics' in col) and ('nonxrag' not in col) and (col[-2:] != '_n')]
        last_saturation_with_nonxrag = [col for col in df_features.columns.values if any([st in col for st in saturation_types]) and ('last_group_metrics' in col) and ('nonxrag' in col) and (col[-2:] != '_n') and ('_first' not in col) and ('_last' not in col)] + last_saturation_xrag_only
        last_attn_features = [col for col in df_features.columns.values if ('attn_last' in col)]
        postllm_saturation_xrag_only = [col for col in df_features.columns.values if any([st in col for st in saturation_types]) and ('mid_metrics' in col or 'last_metrics' in col) and ('nonxrag' not in col) and (col[-2:] != '_n')]
        postllm_saturation_with_nonxrag = [col for col in df_features.columns.values if any([st in col for st in saturation_types]) and ('mid_group_metrics' in col or 'last_group_metrics' in col) and ('nonxrag' in col) and (col[-2:] != '_n') and ('_first' not in col) and ('_last' not in col)] + postllm_saturation_xrag_only
        postllm_attn_features = [col for col in df_features.columns.values if ('attn' in col) and (col[-2:] != '_n')]
        
        def evaluate_feature_set(X, y, cv_splits=5):
            pipe = Pipeline([
                ('scaler', StandardScaler()),
                ('clf', LogisticRegression(max_iter=2000, random_state=42, C=1))
            ])
            cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=42)
            scoring = {'roc_auc': 'roc_auc'}
            cv_results = cross_validate(pipe, X, y, cv=cv, scoring=scoring, return_train_score=False)
            return {
                'auc_mean': cv_results['test_roc_auc'].mean(),
                'auc_std': cv_results['test_roc_auc'].std(),
            }
        
        # Evaluate all feature sets
        features_results = {}
        
        for feat_name, feat_list in [
            ('context', context_features),
            ('preproj', preproj_saturation_features),
            ('postproj', postproj_saturation_features),
            ('prellm', prellm_saturation_features),
            ('mid_xrag', mid_saturation_xrag_only),
            ('mid_joint', mid_saturation_with_nonxrag),
            ('mid_attn', mid_attn_features),
            ('last_xrag', last_saturation_xrag_only),
            ('last_joint', last_saturation_with_nonxrag),
            ('last_attn', last_attn_features),
            ('postllm_xrag', postllm_saturation_xrag_only),
            ('postllm_joint', postllm_saturation_with_nonxrag),
            ('postllm_attn', postllm_attn_features),
        ]:
            if feat_list:
                bad_cols = [c for c in feat_list if df_features[c].apply(lambda x: isinstance(x, dict)).any()]
                if bad_cols:
                    print(feat_name, "has dict columns:", bad_cols)
                if feat_name == 'context' and all(c in df_features.columns for c in feat_list):
                    data = df_features[feat_list + ['overflow_label']].dropna()
                elif feat_name != 'context':
                    data = df_features[feat_list + ['overflow_label']].dropna()
                else:
                    continue
                
                if len(data) > 0:
                    features_results[feat_name] = evaluate_feature_set(data[feat_list].values, data['overflow_label'].values)
        
        all_data[dataset]['features'] = features_results
    
    def fmt(mean, std):
        if mean is None or std is None:
            return "---"
        return f"{mean:.3f} ± {std:.3f}"
    
    def get_probe_metrics(dataset, key):
        for result_type in ['probing_no_query', 'probing_setting1', 'probing_setting2']:
            data = all_data[dataset][result_type]
            if data and key in data:
                result = data[key]
                return {
                    'auc_mean': result.get('auc'),
                    'auc_std': result.get('auc_std'),
                }
        return None
    
    def get_feature_metrics(dataset, key):
        if all_data[dataset]['features'] and key in all_data[dataset]['features']:
            return all_data[dataset]['features'][key]
        return None
    
    # Build DataFrame tables (same structure, no LaTeX)
    def build_table_df(sections_config):
        """Build a single table as (df_display, df_means) for in-notebook display."""
        all_metrics_by_dataset = {d: [] for d in datasets}
        for stage_name, rows in sections_config:
            for feature_name, keys in rows:
                probe_key, feature_key = keys
                for dataset in datasets:
                    m = get_probe_metrics(dataset, probe_key) if probe_key else (get_feature_metrics(dataset, feature_key) if feature_key else None)
                    if m and m.get('auc_mean') is not None:
                        all_metrics_by_dataset[dataset].append(m['auc_mean'])
        max_values = {d: (sorted([v for v in all_metrics_by_dataset[d] if v is not None], reverse=True)[:1] or [None])[0] for d in datasets}
        second_max_values = {}
        for d in datasets:
            vals = sorted([v for v in all_metrics_by_dataset[d] if v is not None], reverse=True)
            second_max_values[d] = vals[1] if len(vals) > 1 else None

        rows_display = []
        rows_means = []
        for stage_name, section_rows in sections_config:
            for row_idx, (feature_name, keys) in enumerate(section_rows):
                probe_key, feature_key = keys
                row_d = {'Stage': stage_name if row_idx == 0 else '', 'Features': feature_name}
                row_m = {'Stage': stage_name if row_idx == 0 else '', 'Features': feature_name}
                for i, dataset in enumerate(datasets):
                    col = data_cols[i]
                    m = get_probe_metrics(dataset, probe_key) if probe_key else (get_feature_metrics(dataset, feature_key) if feature_key else None)
                    if m and m.get('auc_mean') is not None:
                        row_d[col] = fmt(m['auc_mean'], m['auc_std'])
                        row_m[col] = m['auc_mean']
                    else:
                        row_d[col] = '---'
                        row_m[col] = np.nan
                rows_display.append(row_d)
                rows_means.append(row_m)
        return pd.DataFrame(rows_display), pd.DataFrame(rows_means)

    
    # ======== TABLE 1: MAIN RESULTS ========
    # Stage names: Input | Encoding | Post-inference (see STAGE_NAMING_FOR_PAPER.md)
    main_results_sections = [
        ("Pre-compression", [  # or "Pre-compression" if paper is compression-centric
            ("Context", [None, 'context']),
        ]),
        ("Pre-inference", [  # or "Pre-inference"; embedding stage before full LLM
            ("Embedding", ['xrag_preproj+postproj_no_query_linear_sklearn', None]),
            ("Embedding-joint", ['setting1_preproj+postproj_with_preproj_q+postproj_q_linear_sklearn', None]),
            ("Saturation", [None, 'prellm']),
        ]),
        ("Post-inference", [  # or "Post-LLM inference"
            ("Attention", [None, 'postllm_attn']),
            ("Embedding", ['xrag_mid+last_no_query_linear_sklearn', None]),
            ("Embedding-joint", ['setting2_mid+last_with_mid_q+last_q_linear_sklearn', None]),
            ("Saturation", [None, 'postllm_xrag']),
            ("Saturation-joint", [None, 'postllm_joint']),
        ]),
    ]
    
    table1_df, table1_means = build_table_df(main_results_sections)
    
    # ======== TABLE 2: INTERNAL ABLATION ========
    internal_ablation_sections = [
        ("Pre-projection", [
            ("Embedding", ['xrag_preproj_no_query_linear_sklearn', None]),
            ("Embedding-joint", ['setting1_preproj_with_preproj_q_linear_sklearn', None]),
            ("Saturation", [None, 'preproj']),
        ]),
        ("Post-projection", [
            ("Embedding", ['xrag_postproj_no_query_linear_sklearn', None]),
            ("Embedding-joint", ['setting1_postproj_with_postproj_q_linear_sklearn', None]),
            ("Saturation", [None, 'postproj']),
        ]),
        ("Middle layer", [
            ("Attention", [None, 'mid_attn']),
            ("Embedding", ['xrag_mid_no_query_linear_sklearn', None]),
            ("Embedding-joint", ['setting2_mid_with_mid_q_linear_sklearn', None]),
            ("Saturation", [None, 'mid_xrag']),
            ("Saturation-joint", [None, 'mid_joint']),
        ]),
        ("Last layer", [
            ("Attention", [None, 'last_attn']),
            ("Embedding", ['xrag_last_no_query_linear_sklearn', None]),
            ("Embedding-joint", ['setting2_last_with_last_q_linear_sklearn', None]),
            ("Saturation", [None, 'last_xrag']),
            ("Saturation-joint", [None, 'last_joint']),
        ]),
    ]
    
    table2_df, table2_means = build_table_df(internal_ablation_sections)
    
    return (table1_df, table1_means), (table2_df, table2_means)


# Generate tables and display (bold = best, underline = second per dataset column)
(table1_df, table1_means), (table2_df, table2_means) = generate_tables()

display(table1_df.style.apply(lambda _: style_auc_table(table1_df, table1_means), axis=None))

display(table2_df.style.apply(lambda _: style_auc_table(table2_df, table2_means), axis=None))


100
Index(['id', 'gold_answer', 'task_type', 'input_len', 'xrag_pos',
       'mid_layer_index', 'last_layer_index', 'overflow_label', 'context_text',
       'context_perplexity',
       ...
       'attn_last.nonxrag_in_to_xrag_mean_std', 'attn_last.xrag_in_share_mean',
       'attn_last.xrag_in_share_max', 'attn_last.xrag_in_share_min',
       'attn_last.xrag_in_share_std', 'attn_last.xrag_out_entropy_mean',
       'attn_last.xrag_out_entropy_max', 'attn_last.xrag_out_entropy_min',
       'attn_last.xrag_out_entropy_std',
       'attn_last.xrag_out_xrag_vs_nonx_ratio_mean'],
      dtype='object', length=199)


,Stage,Features,TriviaQA
0,Pre-compression,Context,0.519 ± 0.151
1,Pre-inference,Embedding,---
2,,Embedding-joint,0.683 ± 0.067
3,,Saturation,0.608 ± 0.129
4,Post-inference,Attention,0.557 ± 0.171
5,,Embedding,---
6,,Embedding-joint,0.643 ± 0.061
7,,Saturation,0.432 ± 0.105
8,,Saturation-joint,0.406 ± 0.064


,Stage,Features,TriviaQA
0,Pre-projection,Embedding,---
1,,Embedding-joint,0.676 ± 0.076
2,,Saturation,0.594 ± 0.086
3,Post-projection,Embedding,---
4,,Embedding-joint,0.674 ± 0.072
5,,Saturation,0.478 ± 0.158
6,Middle layer,Attention,0.435 ± 0.121
7,,Embedding,---
8,,Embedding-joint,0.640 ± 0.056
9,,Saturation,0.479 ± 0.159
